# Введение в анализ данных. Где живут данные? — notebook для преподавателя

Этот блокнот можно вести как:
1. **slide-like конспект** прямо в Colab;
2. **демонстрационный ноутбук**;
3. основу для раздачи студентам.

> Педагогическая цель пары: не «показать сайты», а научить студента **выбирать, проверять и описывать источник данных до анализа**.


## Как лучше вести этот материал в аудитории

- сначала проговорить, **зачем вообще проверять данные до анализа**;
- показать 3 формы данных: CSV, JSON, API;
- один раз выйти в реальный поиск;
- не перегружать аудиторию программированием;
- в практику отправлять не в полностью свободный поиск, а в **кластеры тематики**.


In [ ]:
import io
import json
from textwrap import dedent

import pandas as pd
import requests
from IPython.display import HTML, Markdown, display

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)


In [ ]:
def card(title, body, color="#274c77", bg="#eef4fa"):
    html = f'''
    <div style="border:1px solid #d8e0ea;border-radius:16px;padding:16px 18px;background:{bg};margin:10px 0;">
        <div style="font-weight:700;color:{color};font-size:18px;margin-bottom:8px;">{title}</div>
        <div style="line-height:1.55;color:#1f2937;">{body}</div>
    </div>
    '''
    display(HTML(html))


In [ ]:
card(
    "Структура пары",
    "<ol>"
    "<li>Поставить кейс: вам дали тему, но не дали данные.</li>"
    "<li>Ввести ключевые понятия: open data, metadata, license, unit of observation.</li>"
    "<li>Показать CSV → JSON → API.</li>"
    "<li>Отправить группы по кластерам в поиск 3 источников.</li>"
    "<li>Вернуть аудиторию к вопросу: почему вы выбрали именно этот набор?</li>"
    "</ol>"
)


## 1. Открытие занятия: кейсовый заход

Вводная реплика:

> «Представьте, что вы аналитик. Вам дали тему исследования, но не дали данные.  
> Что вы будете делать первым делом?»

Цель этой реплики — переключить внимание студентов с «инструментов вообще» на **решение аналитической задачи**.


## 2. Ключевые понятия, которые обязательно проговорить

- **Dataset** — содержательное множество наблюдений.
- **Portal** — место, где публикуются наборы.
- **API** — способ получать данные по запросу.
- **Metadata** — описание данных: publisher, update date, license, schema, coverage.
- **Unit of observation** — что именно представляет одна строка / запись.
- **License / terms of use** — что вам разрешено делать с данными.


> **Замечание для лектора.**  
> Не нужно превращать этот блок в полноценную правовую лекцию.  
> Достаточно сделать лицензию и ограничения частью вопроса «можно ли вообще брать этот источник в работу?»


## 3. Демо 1: табличный формат (CSV)

In [ ]:
csv_text = '''city,year,population,avg_temp,source
Moscow,2023,13100000,6.1,city_statistics
Kazan,2023,1318000,5.8,city_statistics
Novosibirsk,2023,1634000,1.9,city_statistics
Ekaterinburg,2023,1544000,3.4,city_statistics
'''
df_csv = pd.read_csv(io.StringIO(csv_text))
df_csv


### Что спросить аудиторию
- Что здесь является единицей наблюдения?
- Где тут уже видно структуру данных?
- Какие важные метаданные отсутствуют?

### Ожидаемый вывод
Даже если таблица читается легко, без описания источника, даты обновления и условий использования она ещё не становится хорошим аналитическим датасетом.


## 4. Демо 2: JSON как форма данных

In [ ]:
json_text = dedent('''
{
  "dataset": {
    "title": "Urban Mobility Demo",
    "publisher": "Open Transport Lab",
    "updated_at": "2026-02-15",
    "license": "CC BY 4.0",
    "records": [
      {"city": "Moscow", "metro_daily_riders_mln": 6.3, "year": 2025},
      {"city": "Saint Petersburg", "metro_daily_riders_mln": 2.1, "year": 2025},
      {"city": "Kazan", "metro_daily_riders_mln": 0.35, "year": 2025}
    ]
  }
}
''')
payload = json.loads(json_text)
payload


In [ ]:
df_json = pd.DataFrame(payload["dataset"]["records"])
df_json


### Педагогический смысл этого куска
Показать, что:
- данные могут быть структурированы не как таблица;
- метаданные могут жить отдельно от наблюдений;
- аналитику важно уметь различать служебную и содержательную часть ответа.


## 5. Демо 3: API как ещё одно место, где «живут данные»

In [ ]:
url = "https://api.worldbank.org/v2/country/NLD/indicator/SP.POP.TOTL?format=json&per_page=5"
response = requests.get(url, timeout=30)
response.status_code


In [ ]:
api_data = response.json()
api_data[0]


In [ ]:
df_api = pd.DataFrame(api_data[1])[["countryiso3code", "date", "value", "unit", "obs_status"]]
df_api


### Что здесь важно проговорить
- API не страшен: это просто источник, где данные приходят по запросу;
- на этом занятии не нужно массово загонять студентов в Postman;
- достаточно показать идею: **данные могут быть доступны не только через кнопку Download**.


## 6. Реальные стартовые точки поиска

Универсальные ресурсы:
- Kaggle Datasets — https://www.kaggle.com/datasets
- Google Dataset Search — https://datasetsearch.research.google.com/
- Портал открытых данных РФ — https://data.gov.ru/
- World Bank Open Data — https://data.worldbank.org/
- WHO GHO — https://www.who.int/data/gho
- OpenAlex API — https://api.openalex.org

Дополнительно по кластерам:
- IT/AI: Stack Overflow Survey, GH Archive
- Cyber/Infra: CISA KEV, NVD, RIPE Atlas
- Urban/Ecology: NOAA, data.gov
- Business/Humanities: ECB, Europeana, Google Trends


## 7. Как организовать практику на потоке 70 человек

Рекомендуемый способ:
- делим поток на пары/тройки;
- заранее задаём **кластеры**;
- в каждом кластере даём 3–5 стартовых источников;
- все группы делают один и тот же артефакт: **паспорт датасета**.

Так вы снижаете хаос и сохраняете сопоставимость результатов.


In [ ]:
comparison = pd.DataFrame([
    {
        "theme": "",
        "source_name": "",
        "url": "",
        "source_type": "",
        "publisher": "",
        "updated_at": "",
        "license": "",
        "format": "",
        "why_relevant": "",
        "main_risk": "",
    }
    for _ in range(3)
])
comparison


## 8. Готовый шаблон формулировки задания

> Найдите 3 источника данных по вашей теме.  
> Сравните их по владельцу, дате обновления, лицензии, формату и пригодности.  
> Выберите 1 лучший источник.  
> Оформите по нему паспорт датасета и подготовьте мини-объяснение, почему вы выбрали именно его.

Эта формулировка достаточно короткая, но задаёт нужное направление.


In [ ]:
dataset_title = "Название датасета"  #@param {type:"string"}
dataset_url = "https://example.org/dataset"  #@param {type:"string"}
dataset_type = "Портал открытых данных"  #@param ["Портал открытых данных", "Каталог датасетов", "API", "CSV / файл", "Другое"]
dataset_publisher = "Владелец / publisher"  #@param {type:"string"}
dataset_updated = "2026-03-01"  #@param {type:"string"}
dataset_license = "Проверить вручную на странице источника"  #@param {type:"string"}
dataset_format = "CSV"  #@param ["CSV", "JSON", "API / JSON", "XLSX", "XML", "Несколько форматов"]
dataset_unit = "Одна строка = ..."  #@param {type:"string"}
dataset_fields = "id, date, category, value"  #@param {type:"string"}
dataset_task = "Для какой аналитической задачи подходит?"  #@param {type:"string"}
dataset_risks = "Какие ограничения и риски вы видите?"  #@param {type:"string"}

passport_html = f'''
<div style="border:1px solid #d8e0ea;border-radius:18px;overflow:hidden;">
  <div style="background:linear-gradient(135deg,#274c77,#3d6a99);color:white;padding:16px 18px;">
    <div style="font-size:24px;font-weight:700;">{dataset_title}</div>
    <div style="opacity:.92;margin-top:6px;">{dataset_type} · {dataset_format}</div>
  </div>
  <div style="padding:16px 18px;background:white;">
    <p><b>Ссылка:</b> {dataset_url}</p>
    <p><b>Publisher:</b> {dataset_publisher}</p>
    <p><b>Дата обновления:</b> {dataset_updated}</p>
    <p><b>Лицензия:</b> {dataset_license}</p>
    <p><b>Единица наблюдения:</b> {dataset_unit}</p>
    <p><b>Ключевые поля:</b> {dataset_fields}</p>
    <p><b>Подходит для:</b> {dataset_task}</p>
    <p><b>Ограничения и риски:</b> {dataset_risks}</p>
  </div>
</div>
'''
display(HTML(passport_html))


## 9. Что считать хорошей мини-защитой

Студент или группа за 60–90 секунд должны суметь сказать:
1. какая у них тема;
2. какой источник они выбрали;
3. почему именно его;
4. какой риск или limitation заметили.

Если они это могут, значит занятие сработало.


## 10. Домашнее продолжение

Домой не нужно давать новую большую задачу.  
Лучше дать продолжение той же:
- уточнить паспорт;
- добавить описание полей;
- зафиксировать возможный способ импорта;
- сформулировать 2–3 гипотезы для следующего занятия.
